# AlphaZero Chess AI (Self-Contained)
This notebook writes the python files to disk to allow Multiprocessing to work correctly.

In [ ]:
!pip install python-chess torch numpy tqdm

In [ ]:
import torch
print(torch.cuda.is_available())

## 1. Write `utils.py`

In [ ]:
%%writefile utils.py
import chess
import numpy as np
import torch

# 8x8x12 representation
# Channels 0-5: P, N, B, R, Q, K (White)
# Channels 6-11: P, N, B, R, Q, K (Black)
# This is a simplified input (AlphaZero uses historical planes + special planes)
def board_to_tensor(board):
    planes = np.zeros((12, 8, 8), dtype=np.float32)
    
    # Map piece types to channel offsets
    # White pieces: 1-6 -> 0-5
    # Black pieces: 1-6 -> 6-11
    
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece:
            # Row and Col calculation
            # chess.A1 = 0, chess.H8 = 63
            # In numpy, we want [row, col]. 
            # Rank 1 is usually bottom, but for CNN we map naturally.
            # let's map rank 0->7, file 0->7
            rank = chess.square_rank(square)
            file = chess.square_file(square)
            
            piece_type = piece.piece_type # 1=P, 2=N, ... 6=K
            color_offset = 0 if piece.color == chess.WHITE else 6
            
            # channel index
            channel = (piece_type - 1) + color_offset
            planes[channel, rank, file] = 1.0
            
    return planes

def encode_move(move):
    # Simplified encoding: 64 * 64 = 4096 possibilities (from_square -> to_square)
    # This ignores promotion type (always queens or context dependent)
    return move.from_square * 64 + move.to_square

def decode_move(index, board):
    # Decode logic
    from_sq = index // 64
    to_sq = index % 64
    
    # Attempt to create a legal move
    # We must handle promotions. If the move is a promotion, we try to create a Queen promotion.
    move = chess.Move(from_sq, to_sq)
    
    # Check if this strictly corresponds to a promotion
    if chess.square_rank(to_sq) in [0, 7]:
        # Possible promotion
        p = board.piece_at(from_sq)
        if p and p.piece_type == chess.PAWN:
            move.promotion = chess.QUEEN # simple assumption
            
    return move


## 2. Write `model.py`

In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        out = F.relu(out)
        return out

class ChessNet(nn.Module):
    def __init__(self, num_res_blocks=4, num_channels=64):
        super(ChessNet, self).__init__()
        # Input: 12 channels (6 white pieces, 6 black pieces)
        self.start_conv = nn.Conv2d(12, num_channels, kernel_size=3, padding=1, bias=False)
        self.bn_start = nn.BatchNorm2d(num_channels)
        
        self.res_blocks = nn.ModuleList([
            ResidualBlock(num_channels) for _ in range(num_res_blocks)
        ])
        
        # Policy Head
        # Output: 4096 (64*64 moves)
        self.policy_conv = nn.Conv2d(num_channels, 32, kernel_size=1)
        self.policy_bn = nn.BatchNorm2d(32)
        self.policy_fc = nn.Linear(32 * 8 * 8, 4096)
        
        # Value Head
        # Output: 1 scalar (tanh)
        self.value_conv = nn.Conv2d(num_channels, 3, kernel_size=1) # 3 channels roughly
        self.value_bn = nn.BatchNorm2d(3)
        self.value_fc1 = nn.Linear(3 * 8 * 8, 64)
        self.value_fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # x shape: [batch, 12, 8, 8]
        out = F.relu(self.bn_start(self.start_conv(x)))
        
        for block in self.res_blocks:
            out = block(out)
            
        # Policy
        p = F.relu(self.policy_bn(self.policy_conv(out)))
        p = p.flatten(1) # batch, 32*8*8
        p = self.policy_fc(p)
        # Note: We return raw logits (log_softmax is done in loss or implicitly)
        # But MCTS usually expects probabilities. We'll softmax outside or here.
        # Returning logits for numerical stability in training logic usually.
        # But for inference, we want softmax. 
        # Let's return logits for policy, values for value.
        
        # Value
        v = F.relu(self.value_bn(self.value_conv(out)))
        v = v.flatten(1)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v))
        
        return p, v


## 3. Write `mcts.py`

In [ ]:
%%writefile mcts.py
import math
import numpy as np
import torch
import chess
from utils import board_to_tensor, encode_move, decode_move

class MCTSNode:
    def __init__(self, board, parent=None, prior=0):
        self.board = board
        self.parent = parent
        self.children = {} # Map move_index -> MCTSNode
        self.visit_count = 0
        self.value_sum = 0
        self.prior = prior
        self.is_expanded = False
        
    def value(self):
        if self.visit_count == 0:
            return 0
        return self.value_sum / self.visit_count

    def ucb_score(self, child_prior, child_visits, c_puct=1.0):
        # PUCT formula
        # Q + U
        # U = c_puct * P * sqrt(N_parent) / (1 + N_child)
        q_value = 0
        if child_visits > 0:
            # We don't have direct access to child value here without looking up child node
            # But the caller will provide Q usually or we get it from child node
            pass 
        return 0 # Placeholder, logic is in select_child

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

class MCTS:
    def __init__(self, model, device='cuda', c_puct=1.0):
        self.model = model
        self.device = device
        self.c_puct = c_puct
        
    def search(self, board, simulations=50):
        # Create root
        root = MCTSNode(board.copy(), prior=0)
        
        # Expand root first to get priors
        self._expand(root)
        
        for _ in range(simulations):
            node = root
            
            # Select
            while node.is_expanded and len(node.children) > 0:
                node = self._select_child(node)
                
            # Expand & Evaluate
            value = self._expand(node)
            
            # Backpropagate
            self._backpropagate(node, value)
            
        return root

    def _select_child(self, node):
        best_score = -float('inf')
        best_child = None
        
        # Total visits for parent
        total_visits = sum(child.visit_count for child in node.children.values())
        sqrt_total_visits = math.sqrt(total_visits)
        
        for move_idx, child in node.children.items():
            q_value = -child.value() # Value is from perspective of the player moving into that state. 
                                     # child.value() is value for the player WHO JUST MOVED (the opponent of current node)
                                     # So we negate it? 
                                     # Let's clarify:
                                     # Value Head returns value for the CURRENT player in that state.
                                     # Board state passed to NN is "Player to move".
                                     # Value [1] means Player to move wins.
                                     # If I make move M, resulting state S' is Opponent to move.
                                     # NN(S') gives Value' (for Opponent).
                                     # So My Value = -Value'.
                                     
            
            u_value = self.c_puct * child.prior * sqrt_total_visits / (1 + child.visit_count)
            score = q_value + u_value
            
            if score > best_score:
                best_score = score
                best_child = child
                
        return best_child

    def _expand(self, node):
        board = node.board
        
        # Terminal checks
        if board.is_game_over():
            result = board.result()
            if result == '1-0':
                return 1 if board.turn == chess.WHITE else -1
            elif result == '0-1':
                return -1 if board.turn == chess.WHITE else 1
            else:
                return 0 # Draw
        
        # Prepare input
        state_tensor = board_to_tensor(board)
        state_tensor = torch.from_numpy(state_tensor).unsqueeze(0).to(self.device)
        
        self.model.eval()
        with torch.no_grad():
            policy_logits, value = self.model(state_tensor)
            
        policy_probs = torch.softmax(policy_logits, dim=1).cpu().numpy()[0]
        nodes_value = value.item() # Scalar [-1, 1] for current player
        
        # Create children for legal moves
        legal_moves = list(board.legal_moves)
        
        # Mask illegal moves and normalize policy manually 
        # (Usually AlphaZero masks before softmax, but masking after is OK for simple implementation)
        
        policy_sum = 0
        for move in legal_moves:
            move_idx = encode_move(move)
            if move_idx < 4096:
                prob = policy_probs[move_idx]
                policy_sum += prob
                
                # Create Child
                next_board = board.copy()
                next_board.push(move)
                child = MCTSNode(next_board, parent=node, prior=prob)
                node.children[move_idx] = child
        
        # Re-normalize priors
        if policy_sum > 0:
            for child in node.children.values():
                child.prior /= policy_sum
                
        node.is_expanded = True
        return nodes_value

    def _backpropagate(self, node, value):
        curr = node
        while curr is not None:
            curr.visit_count += 1
            curr.value_sum += value
            curr = curr.parent
            value = -value # Flip perspective at each level
            
    def get_action_probs(self, root, temperature=1.0):
        visits = [child.visit_count for child in root.children.values()]
        moves = [move_idx for move_idx in root.children.keys()]
        
        if sum(visits) == 0:
            return [], [] # Should not happen if simulations > 0

        if temperature == 0:
            # Deterministic: pick max visits
            best_idx = np.argmax(visits)
            probs = [0] * len(visits)
            probs[best_idx] = 1
            return moves, probs
        
        # Softmax with temp
        # visits^(1/temp)
        scaled_visits = [v ** (1.0 / temperature) for v in visits]
        sum_scaled = sum(scaled_visits)
        probs = [v / sum_scaled for v in scaled_visits]
        
        return moves, probs


## 4. Write `trainer.py`

In [ ]:
%%writefile trainer.py
import torch
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import chess
from tqdm import tqdm
import os
import random

from model import ChessNet
from mcts import MCTS
from utils import board_to_tensor, encode_move

class ChessDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        # example: (state_tensor, policy_target_vector, value_target)
        # Note: policy_target needs to be expanded to full 4096 size if stored sparsely
        state, policy_dict, value = self.examples[idx]
        
        policy_target = np.zeros(4096, dtype=np.float32)
        for move_idx, prob in policy_dict.items():
            if move_idx < 4096:
                policy_target[move_idx] = prob
                
        return state, policy_target, np.array([value], dtype=np.float32)

import torch.multiprocessing as mp

def run_game_worker(game_idx, total_games, model_state, mcts_sims, device):
    # Re-instantiate model/mcts in the worker process
    # This ensures no complex pickling of CUDA tensors
    try:
        model = ChessNet().to(device)
        model.load_state_dict(model_state)
        model.eval()
        
        mcts = MCTS(model, device)
        board = chess.Board()
        examples = []
        
        while not board.is_game_over():
            # Minimal logging for workers to avoid garbled output
            # Just log every 10 moves or just rely on the final completion log
            # if game_idx % 1 == 0 and len(board.move_stack) % 5 == 0:
            #    print(f"\rWrk {game_idx} | M {len(board.move_stack)}", end="", flush=True)

            root = mcts.search(board, simulations=mcts_sims)
            
            temp = 1.0 if len(board.move_stack) < 30 else 0.1
            moves, probs = mcts.get_action_probs(root, temperature=temp)
            
            policy_dict = {m: p for m, p in zip(moves, probs)}
            state_tensor = board_to_tensor(board)
            examples.append([state_tensor, policy_dict, None])
            
            choice_idx = np.random.choice(len(moves), p=probs)
            best_move_idx = moves[choice_idx]
            
            found_move = None
            for m in board.legal_moves:
                if encode_move(m) == best_move_idx:
                    found_move = m
                    break
            
            if found_move:
                board.push(found_move)
            else:
                break
        
        print(f"Game {game_idx}/{total_games} Finished. Result: {board.result()}")
        
        # Reward assignment
        result = board.result()
        if result == '1-0': reward = 1
        elif result == '0-1': reward = -1
        else: reward = 0
            
        processed_examples = []
        for i, ex in enumerate(examples):
            perspective = 1 if (i % 2 == 0) else -1
            ex[2] = reward * perspective
            processed_examples.append(ex)
            
        return processed_examples
    except Exception as e:
        print(f"Error in worker {game_idx}: {e}")
        return []

class Trainer:
    def __init__(self, device=None):
        if device:
            self.device = device
        else:
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
            
        print(f"Trainer using device: {self.device}")
        if self.device == 'cpu':
            print("WARNING: Training on CPU will be extremely slow. Please install PyTorch with CUDA support.")
            
        self.model = ChessNet().to(self.device)
        try:
            self.model.load_state_dict(torch.load("best_model.pth", weights_only=True))
            print("Resuming training from best_model.pth")
        except:
            print("Starting fresh training.")

        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001, weight_decay=1e-4)
        
        # --- CONFIGURATION FOR HIGH COMPUTE ---
        self.mcts_sims = 400
        self.games_per_iteration = 100
        self.epochs = 20
        self.batch_size = 128
        self.num_workers = 4 # Number of parallel games. Adjust based on VRAM (A100 can handle 8-16 probably)
        # --------------------------------------

    def train(self, iterations=1):
        # Ensure spawn method for CUDA
        try:
            mp.set_start_method('spawn', force=True)
        except RuntimeError:
            pass

        for i in range(iterations):
            print(f"Iteration {i+1}/{iterations}...")
            
            # 1. Parallel Self Play
            # Prepare args for workers
            # We pass the state_dict so they can reload it.
            # (Passing the model directly can verify issues with CUDA pointers)
            model_state = self.model.state_dict()
            # Move state to CPU for pickling
            for k, v in model_state.items():
                model_state[k] = v.cpu()
                
            args_list = []
            for g in range(self.games_per_iteration):
                args_list.append((g+1, self.games_per_iteration, model_state, self.mcts_sims, self.device))
            
            iteration_examples = []
            print(f"Starting {self.games_per_iteration} games with {self.num_workers} workers...")
            
            with mp.Pool(processes=self.num_workers) as pool:
                # Use starmap to pass multiple args
                results = pool.starmap(run_game_worker, args_list)
                
            for res in results:
                iteration_examples.extend(res)
            
            # 2. Train
            print(f"Training on {len(iteration_examples)} positions...")
            dataset = ChessDataset(iteration_examples)
            dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
            
            self.model.train()
            total_loss = 0
            for state, p_target, v_target in tqdm(dataloader, desc="Training"):
                state, p_target, v_target = state.to(self.device), p_target.to(self.device), v_target.to(self.device)
                
                self.optimizer.zero_grad()
                p_out, v_out = self.model(state)
                
                v_loss = F.mse_loss(v_out.view(-1), v_target.view(-1))
                log_probs = F.log_softmax(p_out, dim=1)
                p_loss = -torch.sum(p_target * log_probs) / state.size(0)
                
                loss = v_loss + p_loss
                loss.backward()
                self.optimizer.step()
                
                total_loss += loss.item()
                
            print(f"Avg Loss: {total_loss / len(dataloader):.4f}")
            
            # Save
            torch.save(self.model.state_dict(), "best_model.pth")
            os.makedirs("checkpoints", exist_ok=True)
            torch.save(self.model.state_dict(), f"checkpoints/model_iter_{i+1}.pth")
            print(f"Model saved: best_model.pth & checkpoints/model_iter_{i+1}.pth")

if __name__ == "__main__":
    trainer = Trainer()
    trainer.train(iterations=1)


## 5. Start Training

In [ ]:
from trainer import Trainer

# Initialize and Train
trainer = Trainer()
trainer.train(iterations=50)